In [76]:
import os
import math
import random
import time
import json
from pathlib import Path
from typing import List, Tuple, Dict, Any
from collections import Counter

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
from tqdm import tqdm

In [77]:
# Download required NLTK data
import nltk
required_packages = ["punkt", "wordnet"]
for package in required_packages:
    try:
        resource_path = f"tokenizers/{package}" if package == "punkt" else f"corpora/{package}"
        nltk.data.find(resource_path)
    except LookupError:
        print(f"Downloading {package}...")
        nltk.download(package)

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [78]:
# Set random seeds for reproducibility
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

# Device configuration
COMPUTATION_DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {COMPUTATION_DEVICE}")

Using device: cpu


In [79]:
class ModelConfiguration:
    """Centralized configuration for the image captioning model"""

    # Data paths and settings
    DATASET_ROOT = "/content/RSICD"
    IMAGE_FOLDER = "RSICD_images"
    CAPTION_FILE = "RSICD_captions.txt"

    # Image preprocessing
    IMAGE_SIZE = 224
    SEQUENCE_LENGTH = 22

    # Vocabulary settings
    MINIMUM_WORD_FREQUENCY = 2
    VOCABULARY_SIZE_LIMIT = 10000

    # Training parameters
    BATCH_SIZE = 64
    NUM_WORKERS = 2
    TRAINING_EPOCHS = 3
    LSTM_LEARNING_RATE = 2e-4
    TRANSFORMER_LEARNING_RATE = 2e-4

    # Special tokens
    PADDING_TOKEN = "<pad>"
    START_TOKEN = "<bos>"
    END_TOKEN = "<eos>"
    UNKNOWN_TOKEN = "<unk>"

    # Model architecture parameters
    EMBEDDING_DIMENSION = 512
    HIDDEN_DIMENSION = 512
    RNN_LAYERS = 1

    # Transformer parameters
    MODEL_DIMENSION = 512
    ATTENTION_HEADS = 4
    TRANSFORMER_LAYERS = 2
    DROPOUT_RATE = 0.1

    # Generation settings
    MAX_GENERATION_LENGTH = 24


# Create data directory if it doesn't exist
Path(ModelConfiguration.DATASET_ROOT).mkdir(parents=True, exist_ok=True)
print(f"Dataset root: {ModelConfiguration.DATASET_ROOT}")

Dataset root: /content/RSICD


In [80]:
def load_image_caption_pairs(file_path: str) -> List[Tuple[str, str]]:
    """
    Load image-caption pairs from file
    Creates dummy data if file doesn't exist for testing purposes
    """
    image_caption_pairs = []

    if not os.path.exists(file_path):
        # Create sample data for testing
        print("Creating sample data file...")
        with open(file_path, "w", encoding="utf-8") as f:
            f.write("airport1.jpg\tan airport runway with multiple aircraft parked\n")
            f.write("river2.jpg\ta winding river flowing through agricultural fields\n")
            f.write("urban3.jpg\tdense urban area with buildings and street network\n")

    with open(file_path, "r", encoding="utf-8") as f:
        for line_number, line in enumerate(f, 1):
            line = line.strip()
            if not line:
                continue

            # Handle different separator formats
            if "\t" in line:
                image_name, caption = line.split("\t", 1)
            else:
                parts = line.split(",", 1)
                if len(parts) != 2:
                    print(f"Warning: Skipping malformed line {line_number}")
                    continue
                image_name, caption = parts

            image_caption_pairs.append((image_name.strip(), caption.strip()))

    return image_caption_pairs

In [81]:
def extract_and_prepare_data():
    """Extract data from zip file and prepare train/val/test splits"""
    import zipfile

    # Extract the training data
    zip_path = "/content/drive/MyDrive/Assignment_1/train.csv.zip"
    extract_path = "/content/drive/MyDrive/Assignment_1/"

    try:
        with zipfile.ZipFile(zip_path, "r") as zip_ref:
            zip_ref.extractall(extract_path)
        print("Data extracted successfully")
    except FileNotFoundError:
        print("Zip file not found, will use dummy data")

    # Load and shuffle the data
    data_file = "/content/drive/MyDrive/Assignment_1/train.csv"
    all_pairs = load_image_caption_pairs(data_file)
    random.shuffle(all_pairs)

    # Split into train/validation/test
    train_data = all_pairs[:8000]
    validation_data = all_pairs[8000:9500]
    test_data = all_pairs[9500:10921]

    print(f"Data splits - Train: {len(train_data)}, Val: {len(validation_data)}, Test: {len(test_data)}")
    return train_data, validation_data, test_data

In [82]:
def tokenize_text(text: str) -> List[str]:
    """Tokenize text using NLTK word tokenizer"""
    return nltk.word_tokenize(text.lower())


def create_vocabulary(training_pairs: List[Tuple[str, str]],
                     min_frequency: int = 2,
                     vocab_limit: int = 10000) -> Tuple[Dict[str, int], List[str], Counter]:
    """
    Build vocabulary from training captions
    Returns: word_to_index, index_to_word, word_counts
    """
    special_tokens = [
        ModelConfiguration.PADDING_TOKEN,
        ModelConfiguration.START_TOKEN,
        ModelConfiguration.END_TOKEN,
        ModelConfiguration.UNKNOWN_TOKEN
    ]

    # Count word frequencies
    word_counter = Counter()
    for _, caption in training_pairs:
        word_counter.update(tokenize_text(caption))

    # Filter by minimum frequency and limit vocabulary size
    frequent_words = [word for word, count in word_counter.items() if count >= min_frequency]
    frequent_words = sorted(frequent_words, key=lambda w: (-word_counter[w], w))

    # Limit vocabulary size (accounting for special tokens)
    max_vocab_words = max(0, vocab_limit - len(special_tokens))
    frequent_words = frequent_words[:max_vocab_words]

    # Create final vocabulary
    vocabulary_list = special_tokens + frequent_words
    word_to_index = {word: idx for idx, word in enumerate(vocabulary_list)}

    return word_to_index, vocabulary_list, word_counter


# Download required NLTK data again to ensure availability
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)

# Prepare the data
train_pairs, val_pairs, test_pairs = extract_and_prepare_data()

# Build vocabulary
print(f"Special tokens: {ModelConfiguration.PADDING_TOKEN}, {ModelConfiguration.START_TOKEN}, "
      f"{ModelConfiguration.END_TOKEN}, {ModelConfiguration.UNKNOWN_TOKEN}")

word_to_idx, idx_to_word, word_counts = create_vocabulary(
    train_pairs,
    ModelConfiguration.MINIMUM_WORD_FREQUENCY,
    ModelConfiguration.VOCABULARY_SIZE_LIMIT
)

print(f"Vocabulary preview: {idx_to_word[:10]}")

# Get special token indices
PAD_INDEX = word_to_idx[ModelConfiguration.PADDING_TOKEN]
START_INDEX = word_to_idx[ModelConfiguration.START_TOKEN]
END_INDEX = word_to_idx[ModelConfiguration.END_TOKEN]
UNK_INDEX = word_to_idx[ModelConfiguration.UNKNOWN_TOKEN]

print(f"Vocabulary size: {len(idx_to_word)}")
print(f"Token indices - PAD: {PAD_INDEX}, START: {START_INDEX}, END: {END_INDEX}, UNK: {UNK_INDEX}")

Streaming output truncated to the last 5000 lines.
Data splits - Train: 8000, Val: 1500, Test: 1421
Special tokens: <pad>, <bos>, <eos>, <unk>
Vocabulary preview: ['<pad>', '<bos>', '<eos>', '<unk>', "''", '$', ':', '[', '}', '#']
Vocabulary size: 10000
Token indices - PAD: 0, START: 1, END: 2, UNK: 3


In [83]:
def encode_caption(caption: str, max_length: int = ModelConfiguration.SEQUENCE_LENGTH) -> torch.Tensor:
    """Convert caption text to token indices with special tokens"""
    token_ids = [START_INDEX]
    token_ids.extend([word_to_idx.get(token, UNK_INDEX) for token in tokenize_text(caption)])
    token_ids.append(END_INDEX)

    # Pad or truncate to fixed length
    if len(token_ids) < max_length:
        token_ids.extend([PAD_INDEX] * (max_length - len(token_ids)))
    else:
        token_ids = token_ids[:max_length]
        token_ids[-1] = END_INDEX  # Ensure sequence ends with EOS

    return torch.tensor(token_ids, dtype=torch.long)

In [84]:
def decode_caption(token_indices: List[int]) -> str:
    """Convert token indices back to readable text"""
    words = []
    for idx in token_indices:
        idx = int(idx)
        if idx == END_INDEX:
            break
        if idx in (PAD_INDEX, START_INDEX):
            continue

        word = idx_to_word[idx] if 0 <= idx < len(idx_to_word) else ModelConfiguration.UNKNOWN_TOKEN
        words.append(word)

    return " ".join(words)


# Analyze dataset statistics
caption_lengths = [len(tokenize_text(caption)) for _, caption in train_pairs]
val_oov_count = sum(1 for _, caption in val_pairs
                   for token in tokenize_text(caption)
                   if token not in word_to_idx)
val_total_tokens = sum(1 for _, caption in val_pairs for _ in tokenize_text(caption))

print(f"Caption length statistics - Mean: {np.mean(caption_lengths):.2f}, "
      f"Std: {np.std(caption_lengths):.2f}")
print(f"Validation OOV rate: {(100 * val_oov_count / max(1, val_total_tokens)):.2f}%")


# Image preprocessing pipeline
IMAGENET_NORMALIZATION_MEAN = [0.485, 0.456, 0.406]
IMAGENET_NORMALIZATION_STD = [0.229, 0.224, 0.225]

image_transforms = transforms.Compose([
    transforms.Resize((ModelConfiguration.IMAGE_SIZE, ModelConfiguration.IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_NORMALIZATION_MEAN, IMAGENET_NORMALIZATION_STD),
])

Caption length statistics - Mean: 3886.61, Std: 4687.68
Validation OOV rate: 39.44%


In [85]:
class RemoteSensingDataset(Dataset):
    """Dataset class for remote sensing images and captions"""

    def __init__(self, data_pairs: List[Tuple[str, str]],
                 root_directory: str,
                 image_directory: str,
                 transform_pipeline):
        self.data_pairs = data_pairs
        self.root_path = Path(root_directory)
        self.image_directory = image_directory
        self.transform_pipeline = transform_pipeline

    def __len__(self) -> int:
        return len(self.data_pairs)

    def __getitem__(self, index: int) -> Tuple[torch.Tensor, torch.Tensor]:
        image_filename, caption_text = self.data_pairs[index]
        image_path = self.root_path / self.image_directory / image_filename

        # Load image or create dummy if not found
        if image_path.exists():
            image = Image.open(image_path).convert("RGB")
        else:
            # Create a gray placeholder image for missing files
            image = Image.new("RGB", (ModelConfiguration.IMAGE_SIZE, ModelConfiguration.IMAGE_SIZE), (127, 127, 127))

        processed_image = self.transform_pipeline(image)
        encoded_caption = encode_caption(caption_text)

        return processed_image, encoded_caption

In [86]:
def create_batch_collator(batch):
    """Custom collate function for DataLoader"""
    images, captions = zip(*batch)
    return torch.stack(images, dim=0), torch.stack(captions, dim=0)

In [87]:
# Create datasets and data loaders
train_dataset = RemoteSensingDataset(train_pairs, ModelConfiguration.DATASET_ROOT,
                                   ModelConfiguration.IMAGE_FOLDER, image_transforms)
val_dataset = RemoteSensingDataset(val_pairs, ModelConfiguration.DATASET_ROOT,
                                 ModelConfiguration.IMAGE_FOLDER, image_transforms)
test_dataset = RemoteSensingDataset(test_pairs, ModelConfiguration.DATASET_ROOT,
                                  ModelConfiguration.IMAGE_FOLDER, image_transforms)

train_loader = DataLoader(train_dataset, batch_size=ModelConfiguration.BATCH_SIZE,
                         shuffle=True, num_workers=ModelConfiguration.NUM_WORKERS,
                         collate_fn=create_batch_collator)
val_loader = DataLoader(val_dataset, batch_size=ModelConfiguration.BATCH_SIZE,
                       shuffle=False, num_workers=ModelConfiguration.NUM_WORKERS,
                       collate_fn=create_batch_collator)
test_loader = DataLoader(test_dataset, batch_size=ModelConfiguration.BATCH_SIZE,
                        shuffle=False, num_workers=ModelConfiguration.NUM_WORKERS,
                        collate_fn=create_batch_collator)

print(f"Dataset sizes - Train: {len(train_dataset)}, Val: {len(val_dataset)}, Test: {len(test_dataset)}")


class ImageFeatureExtractor(nn.Module):
    """CNN-based image encoder using pretrained ResNet"""

    def __init__(self):
        super().__init__()
        # Load pretrained ResNet18
        resnet_model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

        # Remove final classification layers
        self.feature_backbone = nn.Sequential(*list(resnet_model.children())[:-2])
        self.global_pool = nn.AdaptiveAvgPool2d((1, 1))
        self.feature_dimension = 512

        # Freeze earlier layers, fine-tune only the last block
        for parameter in self.feature_backbone.parameters():
            parameter.requires_grad = False

        # Enable gradients for the final block
        for parameter in list(self.feature_backbone.children())[-1].parameters():
            parameter.requires_grad = True

    def forward(self, images: torch.Tensor) -> torch.Tensor:
        # Extract features: [batch_size, channels, height, width]
        features = self.feature_backbone(images)
        # Global average pooling: [batch_size, channels]
        pooled_features = self.global_pool(features).view(features.size(0), -1)
        return pooled_features

Dataset sizes - Train: 8000, Val: 1500, Test: 1421


In [88]:
class LSTMCaptionGenerator(nn.Module):
    """LSTM-based caption decoder"""

    def __init__(self, vocabulary_size: int,
                 embedding_dim: int = 512,
                 hidden_dim: int = 512,
                 num_layers: int = 1,
                 padding_idx: int = 0):
        super().__init__()

        self.token_embedding = nn.Embedding(vocabulary_size, embedding_dim, padding_idx=padding_idx)
        self.initial_hidden_projection = nn.Linear(hidden_dim, hidden_dim)
        self.initial_cell_projection = nn.Linear(hidden_dim, hidden_dim)
        self.lstm_layer = nn.LSTM(embedding_dim, hidden_dim, num_layers=num_layers, batch_first=True)
        self.output_projection = nn.Linear(hidden_dim, vocabulary_size)
        self.hidden_dimension = hidden_dim

    def forward(self, image_features: torch.Tensor, target_sequence: torch.Tensor) -> torch.Tensor:
        # Embed target tokens
        embedded_tokens = self.token_embedding(target_sequence)

        # Initialize LSTM state from image features
        initial_hidden = torch.tanh(self.initial_hidden_projection(image_features)).unsqueeze(0)
        initial_cell = torch.tanh(self.initial_cell_projection(image_features)).unsqueeze(0)

        # Generate sequence through LSTM
        lstm_output, _ = self.lstm_layer(embedded_tokens, (initial_hidden, initial_cell))

        # Project to vocabulary logits
        return self.output_projection(lstm_output)

    def generate_greedy(self, image_features: torch.Tensor,
                       max_length: int = 24,
                       start_token: int = START_INDEX,
                       end_token: int = END_INDEX) -> torch.Tensor:
        """Generate captions using greedy decoding"""
        batch_size = image_features.size(0)

        # Initialize LSTM state
        hidden_state = torch.tanh(self.initial_hidden_projection(image_features)).unsqueeze(0)
        cell_state = torch.tanh(self.initial_cell_projection(image_features)).unsqueeze(0)

        # Start with start token
        current_input = torch.full((batch_size, 1), start_token,
                                 dtype=torch.long, device=image_features.device)
        generated_sequence = [current_input]
        lstm_state = (hidden_state, cell_state)

        for _ in range(max_length - 1):
            # Embed current token
            embedded_input = self.token_embedding(current_input)

            # LSTM forward pass
            lstm_out, lstm_state = self.lstm_layer(embedded_input, lstm_state)

            # Get next token probabilities and choose greedily
            logits = self.output_projection(lstm_out[:, -1, :])
            next_token = logits.argmax(dim=-1, keepdim=True)

            generated_sequence.append(next_token)
            current_input = next_token

            # Stop if all sequences generated end token
            if torch.all(next_token.squeeze(-1) == end_token):
                break

        return torch.cat(generated_sequence, dim=1)

In [89]:
class PositionalEncoder(nn.Module):
    """Positional encoding for transformer models"""

    def __init__(self, model_dimension: int, max_sequence_length: int = 512):
        super().__init__()

        # Create positional encoding matrix
        position_encoding = torch.zeros(max_sequence_length, model_dimension)
        position_indices = torch.arange(0, max_sequence_length).float().unsqueeze(1)

        # Create frequency scaling factors
        frequency_factors = torch.exp(
            torch.arange(0, model_dimension, 2).float() *
            (-math.log(10000.0) / model_dimension)
        )

        # Apply sine and cosine functions
        position_encoding[:, 0::2] = torch.sin(position_indices * frequency_factors)
        position_encoding[:, 1::2] = torch.cos(position_indices * frequency_factors)

        # Register as buffer (non-parameter)
        self.register_buffer("position_encoding", position_encoding.unsqueeze(0))

    def forward(self, embedded_tokens: torch.Tensor) -> torch.Tensor:
        sequence_length = embedded_tokens.size(1)
        return embedded_tokens + self.position_encoding[:, :sequence_length]

In [90]:
def create_causal_attention_mask(sequence_length: int) -> torch.Tensor:
    """Create causal mask to prevent attention to future tokens"""
    mask = torch.triu(torch.ones(sequence_length, sequence_length, dtype=torch.bool), diagonal=1)
    return mask

In [91]:
class TransformerCaptionGenerator(nn.Module):
    """Transformer-based caption decoder"""

    def __init__(self, vocabulary_size: int,
                 model_dimension: int = 512,
                 attention_heads: int = 4,
                 num_layers: int = 2,
                 padding_idx: int = 0,
                 image_feature_dim: int = 512,
                 dropout_rate: float = 0.1):
        super().__init__()

        self.token_embedding = nn.Embedding(vocabulary_size, model_dimension, padding_idx=padding_idx)
        self.positional_encoder = PositionalEncoder(model_dimension)

        # Transformer decoder layers
        decoder_layer = nn.TransformerDecoderLayer(
            d_model=model_dimension,
            nhead=attention_heads,
            dropout=dropout_rate,
            batch_first=True
        )
        self.transformer_decoder = nn.TransformerDecoder(decoder_layer, num_layers=num_layers)

        # Image feature processing
        self.image_feature_projection = nn.Linear(image_feature_dim, model_dimension)
        self.layer_normalization = nn.LayerNorm(model_dimension)

        # Output projection
        self.vocabulary_projection = nn.Linear(model_dimension, vocabulary_size)
        self.padding_index = padding_idx

    def forward(self, image_features: torch.Tensor, target_sequence: torch.Tensor) -> torch.Tensor:
        # Process target sequence
        embedded_targets = self.positional_encoder(self.token_embedding(target_sequence))

        # Process image features as memory
        projected_image_features = self.layer_normalization(
            self.image_feature_projection(image_features)
        ).unsqueeze(1)

        # Create attention masks
        sequence_length = target_sequence.size(1)
        causal_mask = create_causal_attention_mask(sequence_length).to(target_sequence.device)
        padding_mask = (target_sequence == self.padding_index)

        # Transformer decoding
        decoder_output = self.transformer_decoder(
            tgt=embedded_targets,
            memory=projected_image_features,
            tgt_mask=causal_mask,
            tgt_key_padding_mask=padding_mask
        )

        return self.vocabulary_projection(decoder_output)

    def generate_greedy(self, image_features: torch.Tensor,
                       max_length: int = 24,
                       start_token: int = START_INDEX,
                       end_token: int = END_INDEX) -> torch.Tensor:
        """Generate captions using greedy decoding"""
        # Process image features as memory
        memory = self.layer_normalization(
            self.image_feature_projection(image_features)
        ).unsqueeze(1)

        # Initialize with start token
        generated_sequence = torch.full(
            (image_features.size(0), 1),
            start_token,
            dtype=torch.long,
            device=image_features.device
        )

        for _ in range(max_length - 1):
            current_length = generated_sequence.size(1)

            # Create causal mask and process current sequence
            causal_mask = create_causal_attention_mask(current_length).to(generated_sequence.device)
            embedded_sequence = self.positional_encoder(self.token_embedding(generated_sequence))

            # Transformer forward pass
            decoder_output = self.transformer_decoder(
                tgt=embedded_sequence,
                memory=memory,
                tgt_mask=causal_mask,
                tgt_key_padding_mask=(generated_sequence == PAD_INDEX)
            )

            # Get next token
            next_token_logits = self.vocabulary_projection(decoder_output[:, -1, :])
            next_token = next_token_logits.argmax(dim=-1, keepdim=True)

            # Append to sequence
            generated_sequence = torch.cat([generated_sequence, next_token], dim=1)

            # Check for end condition
            if torch.all(next_token.squeeze(-1) == end_token):
                break

        return generated_sequence

In [92]:
def calculate_cross_entropy_loss(logits: torch.Tensor, targets: torch.Tensor,
                               ignore_index: int = PAD_INDEX) -> torch.Tensor:
    """Calculate cross-entropy loss ignoring padding tokens"""
    return F.cross_entropy(
        logits.reshape(-1, logits.size(-1)),
        targets.reshape(-1),
        ignore_index=ignore_index
    )


def train_single_epoch(encoder_model: nn.Module,
                      decoder_model: nn.Module,
                      data_loader: DataLoader,
                      optimizer: torch.optim.Optimizer) -> float:
    """Train for one epoch"""
    encoder_model.train()
    decoder_model.train()
    epoch_losses = []

    for batch_images, batch_captions in tqdm(data_loader, desc="Training"):
        batch_images = batch_images.to(COMPUTATION_DEVICE)
        batch_captions = batch_captions.to(COMPUTATION_DEVICE)

        optimizer.zero_grad()

        # Forward pass
        image_features = encoder_model(batch_images)
        caption_logits = decoder_model(image_features, batch_captions[:, :-1])

        # Calculate loss (predict next token)
        loss = calculate_cross_entropy_loss(caption_logits, batch_captions[:, 1:])

        # Backward pass
        loss.backward()
        optimizer.step()

        epoch_losses.append(loss.item())

    return float(np.mean(epoch_losses))

In [93]:
@torch.no_grad()
def validate_single_epoch(encoder_model: nn.Module,
                         decoder_model: nn.Module,
                         data_loader: DataLoader) -> float:
    """Validate for one epoch"""
    encoder_model.eval()
    decoder_model.eval()
    epoch_losses = []

    for batch_images, batch_captions in tqdm(data_loader, desc="Validation"):
        batch_images = batch_images.to(COMPUTATION_DEVICE)
        batch_captions = batch_captions.to(COMPUTATION_DEVICE)

        # Forward pass
        image_features = encoder_model(batch_images)
        caption_logits = decoder_model(image_features, batch_captions[:, :-1])

        # Calculate loss
        loss = calculate_cross_entropy_loss(caption_logits, batch_captions[:, 1:])
        epoch_losses.append(loss.item())

    return float(np.mean(epoch_losses))


def train_lstm_model(num_epochs: int = ModelConfiguration.TRAINING_EPOCHS) -> str:
    """Train the LSTM-based captioning model"""
    print("Training LSTM model...")

    # Initialize models
    encoder = ImageFeatureExtractor().to(COMPUTATION_DEVICE)
    decoder = LSTMCaptionGenerator(
        len(idx_to_word),
        ModelConfiguration.EMBEDDING_DIMENSION,
        ModelConfiguration.HIDDEN_DIMENSION,
        ModelConfiguration.RNN_LAYERS,
        PAD_INDEX
    ).to(COMPUTATION_DEVICE)

    # Setup optimizer
    trainable_parameters = [p for p in list(encoder.parameters()) + list(decoder.parameters())
                          if p.requires_grad]
    optimizer = torch.optim.Adam(trainable_parameters, lr=ModelConfiguration.LSTM_LEARNING_RATE)

    # Training loop
    best_validation_loss = float('inf')
    checkpoint_path = str(Path(ModelConfiguration.DATASET_ROOT) / "lstm_best_model.pt")

    for epoch in range(1, num_epochs + 1):
        train_loss = train_single_epoch(encoder, decoder, train_loader, optimizer)
        val_loss = validate_single_epoch(encoder, decoder, val_loader)

        print(f"[LSTM] Epoch {epoch}: Train Loss = {train_loss:.4f}, Val Loss = {val_loss:.4f}")

        # Save best model
        if val_loss < best_validation_loss:
            best_validation_loss = val_loss
            torch.save({
                "encoder_state": encoder.state_dict(),
                "decoder_state": decoder.state_dict()
            }, checkpoint_path)
            print(f"Saved new best model with validation loss: {val_loss:.4f}")

    return checkpoint_path

In [94]:
def train_transformer_model(num_epochs: int = ModelConfiguration.TRAINING_EPOCHS) -> str:
    """Train the Transformer-based captioning model"""
    print("Training Transformer model...")

    # Initialize models
    encoder = ImageFeatureExtractor().to(COMPUTATION_DEVICE)
    decoder = TransformerCaptionGenerator(
        len(idx_to_word),
        ModelConfiguration.MODEL_DIMENSION,
        ModelConfiguration.ATTENTION_HEADS,
        ModelConfiguration.TRANSFORMER_LAYERS,
        PAD_INDEX,
        image_feature_dim=512,
        dropout_rate=ModelConfiguration.DROPOUT_RATE
    ).to(COMPUTATION_DEVICE)

    # Setup optimizer
    trainable_parameters = [p for p in list(encoder.parameters()) + list(decoder.parameters())
                          if p.requires_grad]
    optimizer = torch.optim.Adam(trainable_parameters, lr=ModelConfiguration.TRANSFORMER_LEARNING_RATE)

    # Training loop
    best_validation_loss = float('inf')
    checkpoint_path = str(Path(ModelConfiguration.DATASET_ROOT) / "transformer_best_model.pt")

    for epoch in range(1, num_epochs + 1):
        train_loss = train_single_epoch(encoder, decoder, train_loader, optimizer)
        val_loss = validate_single_epoch(encoder, decoder, val_loader)

        print(f"[TRANSFORMER] Epoch {epoch}: Train Loss = {train_loss:.4f}, Val Loss = {val_loss:.4f}")

        # Save best model
        if val_loss < best_validation_loss:
            best_validation_loss = val_loss
            torch.save({
                "encoder_state": encoder.state_dict(),
                "decoder_state": decoder.state_dict()
            }, checkpoint_path)
            print(f"Saved new best model with validation loss: {val_loss:.4f}")

    return checkpoint_path

In [95]:
@torch.no_grad()
def load_trained_models(checkpoint_path: str, model_type: str = "lstm") -> Tuple[nn.Module, nn.Module]:
    """Load trained encoder and decoder models"""
    encoder = ImageFeatureExtractor().to(COMPUTATION_DEVICE)

    if model_type.lower() == "lstm":
        decoder = LSTMCaptionGenerator(
            len(idx_to_word),
            ModelConfiguration.EMBEDDING_DIMENSION,
            ModelConfiguration.HIDDEN_DIMENSION,
            ModelConfiguration.RNN_LAYERS,
            PAD_INDEX
        ).to(COMPUTATION_DEVICE)
    else:
        decoder = TransformerCaptionGenerator(
            len(idx_to_word),
            ModelConfiguration.MODEL_DIMENSION,
            ModelConfiguration.ATTENTION_HEADS,
            ModelConfiguration.TRANSFORMER_LAYERS,
            PAD_INDEX,
            image_feature_dim=512,
            dropout_rate=ModelConfiguration.DROPOUT_RATE
        ).to(COMPUTATION_DEVICE)

    # Load saved weights
    checkpoint = torch.load(checkpoint_path, map_location=COMPUTATION_DEVICE)
    encoder.load_state_dict(checkpoint["encoder_state"])
    decoder.load_state_dict(checkpoint["decoder_state"])

    encoder.eval()
    decoder.eval()

    return encoder, decoder

In [96]:
@torch.no_grad()
def generate_sample_captions(checkpoint_path: str,
                           model_type: str = "lstm",
                           num_samples: int = 10,
                           data_split: str = "val") -> List[Tuple[int, str, str]]:
    """Generate sample captions and compare with ground truth"""
    # Select dataset
    dataset = val_dataset if data_split == "val" else test_dataset

    # Load trained models
    encoder, decoder = load_trained_models(checkpoint_path, model_type)

    sample_results = []

    for sample_idx in range(min(num_samples, len(dataset))):
        # Get sample data
        image_tensor, target_caption = dataset[sample_idx]
        image_batch = image_tensor.unsqueeze(0).to(COMPUTATION_DEVICE)

        # Generate caption
        image_features = encoder(image_batch)
        generated_sequence = decoder.generate_greedy(
            image_features,
            ModelConfiguration.MAX_GENERATION_LENGTH,
            START_INDEX,
            END_INDEX
        )

        # Decode captions
        predicted_caption = decode_caption(generated_sequence[0].tolist())
        ground_truth_caption = decode_caption(target_caption.tolist())

        sample_results.append((sample_idx, predicted_caption, ground_truth_caption))

        print(f"Sample {sample_idx + 1}:")
        print(f"  Predicted: {predicted_caption}")
        print(f"  Ground Truth: {ground_truth_caption}")
        print()

    return sample_results


In [97]:
def compute_evaluation_metrics(prediction_reference_pairs: List[Tuple[str, List[str]]]) -> Dict[str, float]:
    """
    Compute BLEU, METEOR and other evaluation metrics
    Args:
        prediction_reference_pairs: List of (prediction, [reference]) tuples
    """
    from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction
    from nltk.translate.meteor_score import meteor_score

    # Prepare data for metrics calculation
    smoothing_function = SmoothingFunction().method4
    hypothesis_tokens = [tokenize_text(pred) for pred, _ in prediction_reference_pairs]
    reference_tokens = [[tokenize_text(ref[0])] for _, ref in prediction_reference_pairs]

    # Calculate BLEU score
    bleu_score = corpus_bleu(reference_tokens, hypothesis_tokens, smoothing_function=smoothing_function)

    # Calculate METEOR scores
    meteor_scores = []
    for (prediction, references) in prediction_reference_pairs:
        meteor_score_value = meteor_score(references, prediction)
        meteor_scores.append(meteor_score_value)

    # Calculate length statistics
    prediction_lengths = [len(tokenize_text(pred)) for pred, _ in prediction_reference_pairs]

    # Count degenerate cases (repeated trigrams)
    degenerate_count = 0
    for prediction, _ in prediction_reference_pairs:
        tokens = tokenize_text(prediction)
        if len(tokens) >= 3:
            # Check for repeated trigrams
            trigrams = [tuple(tokens[i:i+3]) for i in range(len(tokens)-2)]
            if len(set(trigrams)) < len(trigrams):
                degenerate_count += 1

    return {
        "BLEU4": float(bleu_score),
        "METEOR_mean": float(np.mean(meteor_scores)) if meteor_scores else 0.0,
        "length_mean": float(np.mean(prediction_lengths)),
        "length_std": float(np.std(prediction_lengths)),
        "degenerate_percentage": 100.0 * degenerate_count / max(1, len(prediction_reference_pairs))
    }

In [98]:
def run_comprehensive_evaluation(checkpoint_path: str, model_type: str = "lstm") -> Dict[str, float]:
    """Run comprehensive evaluation on test set"""
    print(f"Running comprehensive evaluation for {model_type.upper()} model...")

    # Load models
    encoder, decoder = load_trained_models(checkpoint_path, model_type)

    # Generate captions for entire test set
    prediction_reference_pairs = []

    for batch_images, batch_captions in tqdm(test_loader, desc="Generating captions"):
        batch_images = batch_images.to(COMPUTATION_DEVICE)

        # Generate captions
        image_features = encoder(batch_images)
        generated_sequences = decoder.generate_greedy(
            image_features,
            ModelConfiguration.MAX_GENERATION_LENGTH,
            START_INDEX,
            END_INDEX
        )

        # Process each sample in the batch
        for i in range(batch_images.size(0)):
            predicted_caption = decode_caption(generated_sequences[i].tolist())
            ground_truth_caption = decode_caption(batch_captions[i].tolist())
            prediction_reference_pairs.append((predicted_caption, [ground_truth_caption]))

    # Calculate metrics
    metrics = compute_evaluation_metrics(prediction_reference_pairs)

    print("Evaluation Results:")
    for metric_name, metric_value in metrics.items():
        print(f"  {metric_name}: {metric_value:.4f}")

    return metrics

In [99]:
# Main execution functions
def main_training_pipeline():
    """Main function to train both models"""
    print("Starting Image Captioning Training Pipeline")
    print("=" * 50)

    # Train LSTM model
    print("Phase 1: Training LSTM Model")
    lstm_checkpoint = train_lstm_model(ModelConfiguration.TRAINING_EPOCHS)
    print(f"LSTM model saved to: {lstm_checkpoint}")

    # Train Transformer model
    print("\nPhase 2: Training Transformer Model")
    transformer_checkpoint = train_transformer_model(ModelConfiguration.TRAINING_EPOCHS)
    print(f"Transformer model saved to: {transformer_checkpoint}")

    return lstm_checkpoint, transformer_checkpoint


def main_evaluation_pipeline(lstm_checkpoint: str, transformer_checkpoint: str):
    """Main function to evaluate both models"""
    print("Starting Model Evaluation Pipeline")
    print("=" * 50)

    # Evaluate LSTM model
    print("Evaluating LSTM Model:")
    lstm_metrics = run_comprehensive_evaluation(lstm_checkpoint, "lstm")

    # Generate sample captions for LSTM
    print("\nSample LSTM Captions:")
    lstm_samples = generate_sample_captions(lstm_checkpoint, "lstm", 5, "val")

    print("\n" + "=" * 50)

    # Evaluate Transformer model
    print("Evaluating Transformer Model:")
    transformer_metrics = run_comprehensive_evaluation(transformer_checkpoint, "transformer")

    # Generate sample captions for Transformer
    print("\nSample Transformer Captions:")
    transformer_samples = generate_sample_captions(transformer_checkpoint, "transformer", 5, "val")

    return lstm_metrics, transformer_metrics

In [100]:
"""
Image Captioning Model for Remote Sensing Data
Implements CNN-LSTM and CNN-Transformer architectures for generating captions
"""

'''
# Usage example and main execution
if __name__ == "__main__":
    print("Image Captioning for Remote Sensing Data")
    print("This code implements CNN-LSTM and CNN-Transformer architectures")
    print("for generating captions from remote sensing images.")
    print()

    # Option 1: Train models
    print("To train models, uncomment and run:")
    print("# lstm_ckpt, trans_ckpt = main_training_pipeline()")

    # Option 2: Load existing models and evaluate
    print("\nTo evaluate existing models, set checkpoint paths and run:")
    print("# lstm_path = '/path/to/lstm_best_model.pt'")
    print("# transformer_path = '/path/to/transformer_best_model.pt'")
    print("# lstm_results, trans_results = main_evaluation_pipeline(lstm_path, transformer_path)")

    # Option 3: Generate sample captions
    print("\nTo generate sample captions:")
    print("# samples = generate_sample_captions('model_checkpoint.pt', 'lstm', 10, 'val')")

    print("\n" + "=" * 60)
    print("SETUP COMPLETE - Ready to train or evaluate models!")
    print("=" * 60)'''

Image Captioning for Remote Sensing Data
This code implements CNN-LSTM and CNN-Transformer architectures
for generating captions from remote sensing images.

To train models, uncomment and run:
# lstm_ckpt, trans_ckpt = main_training_pipeline()

To evaluate existing models, set checkpoint paths and run:
# lstm_path = '/path/to/lstm_best_model.pt'
# transformer_path = '/path/to/transformer_best_model.pt'
# lstm_results, trans_results = main_evaluation_pipeline(lstm_path, transformer_path)

To generate sample captions:
# samples = generate_sample_captions('model_checkpoint.pt', 'lstm', 10, 'val')

SETUP COMPLETE - Ready to train or evaluate models!


In [ ]:
# Start training both models
print("Starting training...")
lstm_checkpoint, transformer_checkpoint = main_training_pipeline()

# Evaluate the trained models
print("Starting evaluation...")
lstm_metrics, trans_metrics = main_evaluation_pipeline(lstm_checkpoint, transformer_checkpoint)

Starting training...
Starting Image Captioning Training Pipeline
Phase 1: Training LSTM Model
Training LSTM model...


Training:  81%|████████  | 101/125 [21:23<04:52, 12.18s/it]

Link to the video:
https://drive.google.com/file/d/1YMMPuafWdu6Q_Xh0gI21HwgjFcV9GyTo/view?usp=drive_link